In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 11 报告模块 (report)

提供专业的模型报告生成功能。

**数据说明**: 基于 `hscredit_yyp.xlsx`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path

from hscredit import init_setting
from hscredit.report import ExcelWriter, feature_bin_stats, FeatureAnalyzer

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
坏样本率: 16.70%


## 1. Excel报告生成

In [3]:
# 使用ExcelWriter创建报告
with ExcelWriter('demo_report.xlsx') as writer:
    # 写入数据概览
    summary_df = pd.DataFrame({
        '指标': ['样本数', '特征数', '坏样本数', '坏样本率', '好样本数'],
        '值': [len(df), df.shape[1], df['target'].sum(), 
               f"{df['target'].mean():.2%}", len(df) - df['target'].sum()]
    })
    writer.write_dataframe(summary_df, sheet_name='数据概览')
    
    # 写入特征统计
    stats_df = df.describe().reset_index()
    writer.write_dataframe(stats_df, sheet_name='特征统计')

print("Excel报告已生成: demo_report.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'demo_report.xlsx'

## 2. 特征分箱统计

In [4]:
# 特征分箱统计
feature_name = '中智小牛分C3'

stats = feature_bin_stats(
    df=df,
    feature=feature_name,
    target='target',
    n_bins=10
)

print(f"=== {feature_name} 分箱统计 ===")
display(stats.head(10))

TypeError: feature_bin_stats() missing 1 required positional argument: 'data'

## 3. 特征分析报告

In [5]:
# 使用FeatureAnalyzer进行特征分析
analyzer = FeatureAnalyzer()

features_to_analyze = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1']

for feature in features_to_analyze:
    analyzer.analyze(
        df=df,
        feature=feature,
        target='target'
    )

# 导出分析报告
analyzer.export_to_excel('feature_analysis_report.xlsx')
print("特征分析报告已生成: feature_analysis_report.xlsx")

TypeError: FeatureAnalyzer.analyze() missing 1 required positional argument: 'data'